# 2 · ADLS → Bronze with Auto Loader
Load the CSV files from the external volume into append-only Bronze tables.
Raw values are kept as strings and duplicates are **not** removed in Bronze.

| Volume folder | Bronze table | Files | Rows | Key |
|---|---|---|---|---|
| telemetry_data | bronze_sensor_readings | 11 | 11,653 | reading_id |
| event_data | bronze_failure_events | 1 | 665 | failure_event_id |
| transaction_data | bronze_work_orders | 1 | 335 | work_order_id |

## 1. Config

In [0]:

from pyspark.sql import functions as F

spark.sql("USE capstone_5iot.iot_schema")
VOLUME = "/Volumes/capstone_5iot/iot_schema/iot_vol"
CHK    = f"{VOLUME}/_checkpoints"

# bronze table -> (folder, key, expected files, expected rows)
FEEDS = {
    "bronze_sensor_readings": ("telemetry_data",   "reading_id",       11, 11653),
    "bronze_failure_events":  ("event_data",       "failure_event_id",  1,   665),
    "bronze_work_orders":     ("transaction_data", "work_order_id",     1,   335),
}

## 2. Auto Loader

In [0]:
def autoload(table, folder):
    df = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("cloudFiles.schemaLocation", f"{CHK}/{table}/schema")
          .option("rescuedDataColumn", "_rescued_data")
          .load(f"{VOLUME}/{folder}")
          .withColumn("_source_file",  F.col("_metadata.file_path"))
          .withColumn("_ingestion_ts", F.current_timestamp()))

    (df.writeStream
       .option("checkpointLocation", f"{CHK}/{table}/checkpoint")
       .trigger(availableNow=True)
       .toTable(table)
       .awaitTermination())

for table, (folder, *_) in FEEDS.items():
    autoload(table, folder)
    print(table, spark.table(table).count(), "rows")

## 3. Validation
Row count · files loaded · key not null · no rescued data · duplicates kept (sensor = 41)

In [0]:
results = []
for table, (folder, key, n_files, n_rows) in FEEDS.items():
    df      = spark.table(table)
    cols    = [c for c in df.columns if not c.startswith("_")]
    rows    = df.count()
    files   = df.select("_source_file").distinct().count()
    nulls   = df.filter(F.col(key).isNull()).count()
    rescued = df.filter("_rescued_data IS NOT NULL").count()
    dupes   = rows - df.select(cols).distinct().count()
    exp_dup = 41 if table == "bronze_sensor_readings" else 0
    ok      = rows == n_rows and files == n_files and nulls == 0 and rescued == 0 and dupes == exp_dup
    results.append((table, rows, n_rows, files, nulls, rescued, dupes, "PASS" if ok else "FAIL"))

res = spark.createDataFrame(results, "table string, rows long, expected long, files long, key_nulls long, rescued long, duplicates long, status string")
display(res)
assert res.filter("status = 'FAIL'").count() == 0, "Validation failed"

## 4. Replay test (TC03 / TC06)
Run Auto Loader again – the checkpoint must skip loaded files, so counts must not change.

In [0]:
before = {t: spark.table(t).count() for t in FEEDS}
for table, (folder, *_) in FEEDS.items():
    autoload(table, folder)
after = {t: spark.table(t).count() for t in FEEDS}

print("before:", before)
print("after: ", after)
assert before == after, "Replay added rows"
print("Replay test PASSED")

## 5. Lineage – rows per file

In [0]:
%sql
SELECT regexp_extract(_source_file, '[^/]+$', 0) AS file, COUNT(*) AS rows, MIN(_ingestion_ts) AS ingested_at
FROM bronze_sensor_readings
GROUP BY 1 ORDER BY 1